In [24]:
import os, textstat
from dotenv import load_dotenv
from google.genai import Client, types
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from pydantic import BaseModel, Field
from typing import List, Dict

In [25]:
# retrieve API key
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if api_key is None:
    raise ValueError("Gemini API key not set")

client = Client()

In [26]:
# output -> pydantic model for structured output -> Flesch-Kincaid score and SMOG score (before and after rewriting), simplified text
class ReadabilityEvaluation(BaseModel):
    pre_flesch_kincaid_grade: float = Field(description="Calculated Flesch-Kincaid grade level score before revision")
    post_flesch_kincaid_grade: float = Field(description="Calculated Flesch-Kincaid grade level score after revision")
    pre_smog: float = Field(description="SMOG readability score before revision")
    post_smog: float = Field(description="SMOG readability score after revision")
    revised_text: str = Field(description="Text rewritten at or below target grade level")

In [27]:
# define agent
readability_agent = LlmAgent(
    name="readability_agent",
    model="gemini-3.5-flash-lite",
    description="Rewrites clinical text to make it readable for patients at or below an 8th-grade reading level",
    instruction="""
        You are an expert at evaluating the readability of clinical text and rewriting the text to be more readable.
        Evaluate the input text's readability using the Flesch-Kincaid grade level and SMOG metrics.
        Then, if the reading level required for the text is above the target grade level, rewrite the text
        to make it readable at the target grade level or below while maintaining the text's original meaning.
    
        Follow these guidelines:
        1) Do not insert new information, change the meaning of existing information, or remove significant pieces of information
        from the input text.
        2) You can replace complex medical terms with plain language equivalents or keep the medical term followed by
        a plain language definition.
        3) Your output must match the defined schema.
    """,
    output_schema=ReadabilityEvaluation,
    output_key="readability_result"
)

In [31]:
# run agent
session_service = InMemorySessionService()

runner = Runner(
    app_name="FaithfulMed",
    agent=readability_agent,
    session_service=session_service
)

In [32]:
# test agent
async def test_readability_agent(simplified_text: str, target_grade: int = 8):
    
    # create session
    session = await session_service.create_session(
        app_name="FaithfulMed",
        user_id="patient_01",
        session_id="eval_run_001"
    )

    # create query
    user_message = types.Content(
        role="user",
        parts=[types.Part(text=f"Rewrite the following clinical text at the {target_grade}th-grade level:\n\n{simplified_text}")]
    )

    # run agent stream
    events = runner.run_async(
        session_id=session.id,
        user_id=session.user_id,
        new_message=user_message
    )

    final_response_text = ""
    async for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            final_response_text = event.content.parts[0].text

    # access parsed Pydantic object from session state
    current_session = await session_service.get_session(
        app_name="FaithfulMed",
        user_id=session.user_id,
        session_id=session.id
    )

    # get results
    evaluation_result = current_session.state.get("readability_result")

    if isinstance(evaluation_result, ReadabilityEvaluation):
        output = evaluation_result
    else:
        # if output is still string
        output = ReadabilityEvaluation.model_validate_json(final_response_text)
    
    print("Simplified Output: ")
    print(output.revised_text)
    
    print("\nLLM-Computed Scores Before Revision: ")
    print(f"Flesch-Kincaid grade level: {output.pre_flesch_kincaid_grade:.2f}")
    print(f"SMOG index: {output.pre_smog:.2f}")
    
    print("\nLLM-Computed Scores After Revision: ")
    print(f"Flesch-Kincaid grade level: {output.post_flesch_kincaid_grade:.2f}")
    print(f"SMOG index: {output.post_smog:.2f}")
    
    # verify textstat metrics w/ Textstat Python library
    computed_flesch_kincaid = textstat.flesch_kincaid_grade(output.revised_text)
    computed_smog = textstat.smog_index(output.revised_text)
    
    print("\nTextstat Scores After Revision: ")
    print(f"Computed Flesch-Kincaid grade level: {computed_flesch_kincaid:.2f}")
    print(f"Computed SMOG index: {computed_smog:.2f}")

In [33]:
# independent agent testing on sample text (generated by Gemini)
gemini_example = """
The patient presented for routine evaluation. Vital signs reveal systolic blood pressure 
of 126 mmHg and diastolic blood pressure of 69 mmHg. Complete blood count demonstrated 
leukocyte count of 8.9 x 10^3/uL, hemoglobin concentration of 15.4 g/dL, and hematocrit 
of 46.5%. Prophylactic immunization against Hepatitis B was administered intramuscularly. 
Medication reconciliation was completed with no adverse contraindications noted.
"""

await test_readability_agent(gemini_example)

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


Simplified Output: 
You came in for a regular check-up. Your blood pressure was 126 over 69. Your blood tests showed a normal white blood cell count of 8.9, a hemoglobin level of 15.4, and a hematocrit of 46.5%. You received a Hepatitis B shot in your muscle to help prevent infection. We reviewed your medicines and found no bad issues or reasons not to take them.

LLM-Computed Scores Before Revision: 
Flesch-Kincaid grade level: 12.50
SMOG index: 14.10

LLM-Computed Scores After Revision: 
Flesch-Kincaid grade level: 7.20
SMOG index: 7.80

Textstat Scores After Revision: 
Computed Flesch-Kincaid grade level: 4.39
Computed SMOG index: 8.42


**Important note**: Investigate discrepancy between computed Flesch-Kincaid grade level and post-revision score given by LLM.

In [ ]:
mtsamples_1 = """
SUBJECTIVE:,  This 23-year-old white female presents with complaint of allergies.  She used to have allergies when she lived in Seattle but she thinks they are worse here.  In the past, she has tried Claritin, and Zyrtec.  Both worked for short time but then seemed to lose effectiveness.  She has used Allegra also.  She used that last summer and she began using it again two weeks ago.  It does not appear to be working very well.  She has used over-the-counter sprays but no prescription nasal sprays.  She does have asthma but doest not require daily medication for this and does not think it is flaring up.,MEDICATIONS: , Her only medication currently is Ortho Tri-Cyclen and the Allegra.,ALLERGIES: , She has no known medicine allergies.,OBJECTIVE:,Vitals:  Weight was 130 pounds and blood pressure 124/78.,HEENT:  Her throat was mildly erythematous without exudate.  Nasal mucosa was erythematous and swollen.  Only clear drainage was seen.  TMs were clear.,Neck:  Supple without adenopathy.,Lungs:  Clear.,ASSESSMENT:,  Allergic rhinitis.,PLAN:,1.  She will try Zyrtec instead of Allegra again.  Another option will be to use loratadine.  She does not think she has prescription coverage so that might be cheaper.,2.  Samples of Nasonex two sprays in each nostril given for three weeks.  A prescription was written as well.
"""
test_readability_agent(mtsamples_1)

In [ ]:
mtsamples_2 = """
PAST MEDICAL HISTORY:, He has difficulty climbing stairs, difficulty with airline seats, tying shoes, used to public seating, and lifting objects off the floor.  He exercises three times a week at home and does cardio.  He has difficulty walking two blocks or five flights of stairs.  Difficulty with snoring.  He has muscle and joint pains including knee pain, back pain, foot and ankle pain, and swelling.  He has gastroesophageal reflux disease.,PAST SURGICAL HISTORY:, Includes reconstructive surgery on his right hand 13 years ago.  ,SOCIAL HISTORY:, He is currently single.  He has about ten drinks a year.  He had smoked significantly up until several months ago.  He now smokes less than three cigarettes a day.,FAMILY HISTORY:, Heart disease in both grandfathers, grandmother with stroke, and a grandmother with diabetes.  Denies obesity and hypertension in other family members.,CURRENT MEDICATIONS:, None.,ALLERGIES:,  He is allergic to Penicillin.,MISCELLANEOUS/EATING HISTORY:, He has been going to support groups for seven months with Lynn Holmberg in Greenwich and he is from Eastchester, New York and he feels that we are the appropriate program.  He had a poor experience with the Greenwich program.  Eating history, he is not an emotional eater.  Does not like sweets.  He likes big portions and carbohydrates.  He likes chicken and not steak.  He currently weighs 312 pounds.  Ideal body weight would be 170 pounds.  He is 142 pounds overweight.  If ,he lost 60% of his excess body weight that would be 84 pounds and he should weigh about 228.,REVIEW OF SYSTEMS: ,Negative for head, neck, heart, lungs, GI, GU, orthopedic, and skin.  Specifically denies chest pain, heart attack, coronary artery disease, congestive heart failure, arrhythmia, atrial fibrillation, pacemaker, high cholesterol, pulmonary embolism, high blood pressure, CVA, venous insufficiency, thrombophlebitis, asthma, shortness of breath, COPD, emphysema, sleep apnea, diabetes, leg and foot swelling, osteoarthritis, rheumatoid arthritis, hiatal hernia, peptic ulcer disease, gallstones, infected gallbladder, pancreatitis, fatty liver, hepatitis, hemorrhoids, rectal bleeding, polyps, incontinence of stool, urinary stress incontinence, or cancer.  Denies cellulitis, pseudotumor cerebri, meningitis, or encephalitis.,PHYSICAL EXAMINATION:, He is alert and oriented x 3.  Cranial nerves II-XII are intact.  Afebrile.  Vital Signs are stable.
"""

test_readability_agent(mtsamples_2)

In [ ]:
mtsamples_3 = """
HISTORY OF PRESENT ILLNESS: , I have seen ABC today.  He is a very pleasant gentleman who is 42 years old, 344 pounds.  He is 5'9".  He has a BMI of 51.  He has been overweight for ten years since the age of 33, at his highest he was 358 pounds, at his lowest 260.  He is pursuing surgical attempts of weight loss to feel good, get healthy, and begin to exercise again.  He wants to be able to exercise and play volleyball.  Physically, he is sluggish.  He gets tired quickly.  He does not go out often.  When he loses weight he always regains it and he gains back more than he lost.  His biggest weight loss is 25 pounds and it was three months before he gained it back.  He did six months of not drinking alcohol and not taking in many calories.  He has been on multiple commercial weight loss programs including Slim Fast for one month one year ago and Atkin's Diet for one month two years ago.,PAST MEDICAL HISTORY: , He has difficulty climbing stairs, difficulty with airline seats, tying shoes, used to public seating, difficulty walking, high cholesterol, and high blood pressure.  He has asthma and difficulty walking two blocks or going eight to ten steps.  He has sleep apnea and snoring.  He is a diabetic, on medication.  He has joint pain, knee pain, back pain, foot and ankle pain, leg and foot swelling.  He has hemorrhoids.,PAST SURGICAL HISTORY: , Includes orthopedic or knee surgery.,SOCIAL HISTORY: , He is currently single.  He drinks alcohol ten to twelve drinks a week, but does not drink five days a week and then will binge drink.  He smokes one and a half pack a day for 15 years, but he has recently stopped smoking for the past two weeks.,FAMILY HISTORY: , Obesity, heart disease, and diabetes.  Family history is negative for hypertension and stroke.,CURRENT MEDICATIONS:,  Include Diovan, Crestor, and Tricor.,MISCELLANEOUS/EATING HISTORY:  ,He says a couple of friends of his have had heart attacks and have had died.  He used to drink everyday, but stopped two years ago.  He now only drinks on weekends.  He is on his second week of Chantix, which is a medication to come off smoking completely.  Eating, he eats bad food.  He is single.  He eats things like bacon, eggs, and cheese, cheeseburgers, fast food, eats four times a day, seven in the morning, at noon, 9 p.m., and 2 a.m.  He currently weighs 344 pounds and 5'9".  His ideal body weight is 160 pounds.  He is 184 pounds overweight.  If he lost 70% of his excess body weight that would be 129 pounds and that would get him down to 215.,REVIEW OF SYSTEMS: , Negative for head, neck, heart, lungs, GI, GU, orthopedic, or skin.  He also is positive for gout.  He denies chest pain, heart attack, coronary artery disease, congestive heart failure, arrhythmia, atrial fibrillation, pacemaker, pulmonary embolism, or CVA.  He denies venous insufficiency or thrombophlebitis.  Denies shortness of breath, COPD, or emphysema.  Denies thyroid problems, hip pain, osteoarthritis, rheumatoid arthritis, GERD, hiatal hernia, peptic ulcer disease, gallstones, infected gallbladder, pancreatitis, fatty liver, hepatitis, rectal bleeding, polyps, incontinence of stool, urinary stress incontinence, or cancer.  He denies cellulitis, pseudotumor cerebri, meningitis, or encephalitis.,PHYSICAL EXAMINATION:  ,He is alert and oriented x 3.  Cranial nerves II-XII are intact.  Neck is soft and supple.  Lungs:  He has positive wheezing bilaterally.  Heart is regular rhythm and rate.  His abdomen is soft.  Extremities:  He has 1+ pitting edema.,IMPRESSION/PLAN:,  I have explained to him the risks and potential complications of laparoscopic gastric bypass in detail and these include bleeding, infection, deep venous thrombosis, pulmonary embolism, leakage from the gastrojejuno-anastomosis, jejunojejuno-anastomosis, and possible bowel obstruction among other potential complications.  He understands.  He wants to proceed with workup and evaluation for laparoscopic Roux-en-Y gastric bypass.  He will need to get a letter of approval from Dr. XYZ.  He will need to see a nutritionist and mental health worker.  He will need an upper endoscopy by either Dr. XYZ.  He will need to go to Dr. XYZ as he previously had a sleep study.  We will need another sleep study.  He will need H. pylori testing, thyroid function tests, LFTs, glycosylated hemoglobin, and fasting blood sugar.  After this is performed, we will submit him for insurance approval.
"""

test_readability_agent(mtsamples_3)

In [ ]:
mtsamples_4 = """
2-D M-MODE: , ,1.  Left atrial enlargement with left atrial diameter of 4.7 cm.,2.  Normal size right and left ventricle.,3.  Normal LV systolic function with left ventricular ejection fraction of 51%.,4.  Normal LV diastolic function.,5.  No pericardial effusion.,6.  Normal morphology of aortic valve, mitral valve, tricuspid valve, and pulmonary valve.,7.  PA systolic pressure is 36 mmHg.,DOPPLER: , ,1.  Mild mitral and tricuspid regurgitation.,2.  Trace aortic and pulmonary regurgitation.
"""

test_readability_agent(mtsamples_4)

In [ ]:
mtsamples_5 = """
1.  The left ventricular cavity size and wall thickness appear normal.  The wall motion and left ventricular systolic function appears hyperdynamic with estimated ejection fraction of 70% to 75%.  There is near-cavity obliteration seen.  There also appears to be increased left ventricular outflow tract gradient at the mid cavity level consistent with hyperdynamic left ventricular systolic function.  There is abnormal left ventricular relaxation pattern seen as well as elevated left atrial pressures seen by Doppler examination.,2.  The left atrium appears mildly dilated.,3.  The right atrium and right ventricle appear normal.,4.  The aortic root appears normal.,5.  The aortic valve appears calcified with mild aortic valve stenosis, calculated aortic valve area is 1.3 cm square with a maximum instantaneous gradient of 34 and a mean gradient of 19 mm.,6.  There is mitral annular calcification extending to leaflets and supportive structures with thickening of mitral valve leaflets with mild mitral regurgitation.,7.  The tricuspid valve appears normal with trace tricuspid regurgitation with moderate pulmonary artery hypertension.  Estimated pulmonary artery systolic pressure is 49 mmHg.  Estimated right atrial pressure of 10 mmHg.,8.  The pulmonary valve appears normal with trace pulmonary insufficiency.,9.  There is no pericardial effusion or intracardiac mass seen.,10.  There is a color Doppler suggestive of a patent foramen ovale with lipomatous hypertrophy of the interatrial septum.,11.  The study was somewhat technically limited and hence subtle abnormalities could be missed from the study.,
"""

test_readability_agent(mtsamples_5)

In [ ]:
mtsamples_6 = """
PREOPERATIVE DIAGNOSIS: , Morbid obesity.,POSTOPERATIVE DIAGNOSIS:  ,Morbid obesity.,PROCEDURE: , Laparoscopic antecolic antegastric Roux-en-Y gastric bypass with EEA anastomosis.,ANESTHESIA: , General with endotracheal intubation.,INDICATION FOR PROCEDURE: , This is a 30-year-old female, who has been overweight for many years.  She has tried many different diets, but is unsuccessful.  She has been to our Bariatric Surgery Seminar, received some handouts, and signed the consent.  The risks and benefits of the procedure have been explained to the patient.,PROCEDURE IN DETAIL:  ,The patient was taken to the operating room and placed supine on the operating room table.  All pressure points were carefully padded.  She was given general anesthesia with endotracheal intubation.  SCD stockings were placed on both legs.  Foley catheter was placed for bladder decompression.  The abdomen was then prepped and draped in standard sterile surgical fashion.  Marcaine was then injected through umbilicus.  A small incision was made.  A Veress needle was introduced into the abdomen.  CO2 insufflation was done to a maximum pressure of 15 mmHg.  A 12-mm VersaStep port was placed through the umbilicus.  I then placed a 5-mm port just anterior to the midaxillary line and just subcostal on the right side.  I placed another 5-mm port in the midclavicular line just subcostal on the right side, a few centimeters below and medial to that, I placed a 12-mm VersaStep port.  On the left side, just anterior to the midaxillary line and just subcostal, I placed a 5-mm port.  A few centimeters below and medial to that, I placed a 15-mm port.  I began by lifting up the omentum and identifying the transverse colon and lifting that up and thereby identifying my ligament of Treitz.  I ran the small bowel down approximately 40 cm and divided the small bowel with a white load GIA stapler.  I then divided the mesentery all the way down to the base of the mesentery with a LigaSure device.  I then ran the distal bowel down, approximately 100 cm, and at 100 cm, I made a hole at the antimesenteric portion of the Roux limb and a hole in the antimesenteric portion of the duodenogastric limb, and I passed a 45 white load stapler and fired a stapler creating a side-to-side anastomosis.  I reapproximated the edges of the defect.  I lifted it up and stapled across it with another white load stapler.  I then closed the mesenteric defect with interrupted Surgidac sutures.  I divided the omentum all the way down to the colon in order to create a passageway for my small bowel to go antecolic.  I then put the patient in reverse Trendelenburg.  I placed a liver retractor, identified, and dissected the angle of His.  I then dissected on the lesser curve, approximately 2.5 cm below the gastroesophageal junction, and got into a lesser space.  I fired transversely across the stomach with a 45 blue load stapler.  I then used two fires of the 60 blue load with SeamGuard to go up into my angle of His, thereby creating my gastric pouch.  I then made a hole at the base of the gastric pouch and had Anesthesia remove the bougie and place the OG tube connected to the anvil.  I pulled the anvil into place, and I then opened up my 15-mm port site and passed my EEA stapler.  I passed that in the end of my Roux limb and had the spike come out antimesenteric.  I joined the spike with the anvil and fired a stapler creating an end-to-side anastomosis, then divided across the redundant portion of my Roux limb with a white load GI stapler, and removed it with an Endocatch bag.  I put some additional 2-0 Vicryl sutures in the anastomosis for further security.  I then placed a bowel clamp across the bowel.  I went above and passed an EGD scope into the mouth down to the esophagus and into the gastric pouch.  I distended gastric pouch with air.  There was no air leak seen.  I could pass the scope easily through the anastomosis.  There was no bleeding seen through the scope.  We closed the 15-mm port site with interrupted 0 Vicryl suture utilizing Carter-Thomason.  I copiously irrigated out that incision with about 2 L of saline.  I then closed the skin of all incisions with running Monocryl.  Sponge, instrument, and needle counts were correct at the end of the case.  The patient tolerated the procedure well without any complications.
"""
test_readability_agent(mtsamples_6)

In [ ]:
mtsamples_7 = """
PREOPERATIVE DIAGNOSES:,1.  Deformity, right breast reconstruction.,2.  Excess soft tissue, anterior abdomen and flank.,3.  Lipodystrophy of the abdomen.,POSTOPERATIVE DIAGNOSES:,1.  Deformity, right breast reconstruction.,2.  Excess soft tissue, anterior abdomen and flank.,3.  Lipodystrophy of the abdomen.,PROCEDURES:,1.  Revision, right breast reconstruction.,2.  Excision, soft tissue fullness of the lateral abdomen and flank.,3.  Liposuction of the supraumbilical abdomen.,ANESTHESIA: , General.,INDICATION FOR OPERATION:,  The patient is a 31-year-old white female who previously has undergone latissimus dorsi flap and implant, breast reconstruction.  She now had lateralization of the implant with loss of medial fullness for which she desired correction.  It was felt that mobilization of the implant medially would provide the patient significant improvement and this was discussed with the patient at length.  The patient also had a small dog ear in the flank area on the right from the latissimus flap harvest, which was to be corrected.  She had also had liposuction of the periumbilical and infraumbilical abdomen with desire to have great improvement superiorly, was felt to be a candidate for such.  The above-noted procedure was discussed with the patient in detail.  The risks, benefits and potential complications were discussed.  She was marked in the upright position and then taken to the operating room for the above-noted procedure.,OPERATIVE PROCEDURE: , The patient was taken to the operating room and placed in the supine position.  Following adequate induction of general LMA anesthesia, the chest and abdomen was prepped and draped in the usual sterile fashion.  The supraumbilical abdomen was then injected with a solution of 5% lidocaine with epinephrine, as was the dog ear.  At this time, the superior central scar was then excised, dissection continued through the subcutaneous tissue, the underlying latissimus muscle until the capsule of the implant was reached.  This was then opened.  The implant was removed and placed on the back table in antibiotic solution.  Using Bovie cautery, the medial capsule was released and undermining was then performed with release of the muscle to the level of the proposed medial projection of the breast.  The inframammary fold medially was secured with 2-0 PDS suture to create greater takeoff point at this level which in the upright position and using a sizer produced a good form.  The lateral pocket was diminished by series of 2-0 PDS suture to provide medialization of the implant.  The implant was then placed back into the submuscular pocket with much improved positioning and medial fullness.  With this completed, the implant was again removed, antibiotic irrigation was performed.  A drain was placed and brought out through a separate inferior stab wound incision and hemostasis was confirmed.  The implant was then replaced and the wound was then closed in layers using 2-0 PDS running suture on the muscle and 3-0 Monocryl Dermabond subcuticular sutures.  The 2.5 cm dog ear was then excised into and including the subcutaneous tissue, even contouring was achieved and this was closed with two layers using 3-0 Monocryl suture.  Using a #3 cannula, a superior umbilical incision, liposuction was carried out into the supraumbilical abdomen, removing approximately 40 to 50 mL of fat with improved supraumbilical contours.  This was closed with 6-0 Prolene suture.  The patient was placed in a compressive garment after treating the incision with Dermabond, Steri-Strips and antibiotic ointment around the drain site and umbilicus.  A Kerlix dressing and a surgical bra was placed to the chest area.  A compressive garment was placed.  The patient was then aroused from anesthesia, extubated, and taken to the recovery room in stable condition.  Sponge, needle, lap, instrument counts were all correct.  The patient tolerated the procedure well.  There were no complications.  The estimated blood loss was approximately 25 mL.
"""

test_readability_agent(mtsamples_7)

In [ ]:
mtsamples_8 = """
2-D ECHOCARDIOGRAM,Multiple views of the heart and great vessels reveal normal intracardiac and great vessel relationships.  Cardiac function is normal.  There is no significant chamber enlargement or hypertrophy.  There is no pericardial effusion or vegetations seen.  Doppler interrogation, including color flow imaging, reveals systemic venous return to the right atrium with normal tricuspid inflow.  Pulmonary outflow is normal at the valve.  Pulmonary venous return is to the left atrium.  The interatrial septum is intact.  Mitral inflow and ascending aorta flow are normal.  The aortic valve is trileaflet.  The coronary arteries appear to be normal in their origins.  The aortic arch is left-sided and patent with normal descending aorta pulsatility.
"""

test_readability_agent(mtsamples_8)

In [ ]:
mtsamples_9 = """
PREOPERATIVE DIAGNOSIS: , Lipodystrophy of the abdomen and thighs.,POSTOPERATIVE DIAGNOSIS:,  Lipodystrophy of the abdomen and thighs.,OPERATION: , Suction-assisted lipectomy.,ANESTHESIA:,  General.,FINDINGS AND PROCEDURE:,  With the patient under satisfactory general endotracheal anesthesia, the entire abdomen, flanks, perineum, and thighs to the knees were prepped and draped circumferentially in sterile fashion.  After this had been completed, a #15 blade was used to make small stab wounds in the lateral hips, the pubic area, and upper edge of the umbilicus.  Through these small incisions, a cannula was used to infiltrate lactated Ringers with 1000 cc was infiltrated initially into the abdomen.  A 3 and 4-mm cannulas were then used to carry out the liposuction of the abdomen removing a total of 1100 cc of aspirate, which was mostly fat, little fluid, and blood.  Attention was then directed to the thighs both inner and outer.  A total of 1000 cc was infiltrated in both lateral thighs only about 50 cc in the medial thighs.  After this had been completed, 3 and 4-mm cannulas were used to suction 650 cc from each side, approximately 50 cc in the inner thigh and 600 on each lateral thigh.  The patient tolerated the procedure very well.  All of this aspirate was mostly fat with little fluid and very little blood.  Wounds were cleaned and steri-stripped and dressing of ABD pads and ***** was then applied.  The patient tolerated the procedure very well and was sent to the recovery room in good condition.
"""

test_readability_agent(mtsamples_9)

In [ ]:
mtsamples_10 = """
DESCRIPTION:,1.  Normal cardiac chambers size.,2.  Normal left ventricular size.,3.  Normal LV systolic function.  Ejection fraction estimated around 60%.,4.  Aortic valve seen with good motion.,5.  Mitral valve seen with good motion.,6.  Tricuspid valve seen with good motion.,7.  No pericardial effusion or intracardiac masses.,DOPPLER:,1.  Trace mitral regurgitation.,2.  Trace tricuspid regurgitation.,IMPRESSION:,1.  Normal LV systolic function.,2.  Ejection fraction estimated around 60%.,
"""

test_readability_agent(mtsamples_10)